In [103]:
import pandas as pd
import re

unlabeled = pd.read_csv("dataset/unlabeled_data.csv")

words = set()

for r in unlabeled["text"]:
    words.update(r.split())

output = pd.DataFrame(columns=["subtaskID", "datapointID", "answer"])
output.loc[0] = [1, 0, len(words)]
print(len(words))

3529


In [104]:
from gensim.utils import simple_preprocess
from collections import Counter

train = pd.read_csv("dataset/train_data.csv")
test = pd.read_csv("dataset/test_data.csv")

texts = pd.concat([unlabeled["text"], test["text"], train["text"]], ignore_index=True)

tokenized = [simple_preprocess(s) for s in texts]

tokenized

[['hwygcitunxy',
  'txaqhsbspp',
  'wppyqfewq',
  'efxtky',
  'sbafp',
  'tzdfeb',
  'wppyqfewq'],
 ['efxtky',
  'hhumxi',
  'fjfuhwt',
  'fdfpxccspynq',
  'wppyqfewq',
  'wppyqfewq',
  'sbafp'],
 ['efxtky',
  'xpylmbi',
  'wppyqfewq',
  'laeoxdw',
  'glhwq',
  'rjeegtrkdlo',
  'qtgwhg',
  'xpylmbi'],
 ['eufs',
  'tfusmgrcm',
  'ehdho',
  'jzinuxcc',
  'qiyicseqlpa',
  'wppyqfewq',
  'kaoj'],
 ['sbafp',
  'mqndwkadzsw',
  'ffatkgedh',
  'qbaymfaw',
  'wppyqfewq',
  'dznpbavcbzcxzg'],
 ['tq', 'sbafp', 'grxusd', 'joehxkmdl', 'xpylmbi', 'ophrmihi', 'jpdyavvikr'],
 ['xpylmbi',
  'rmahhwbggu',
  'xpylmbi',
  'jyilrzwyjsvazpv',
  'xpylmbi',
  'hqhxss',
  'llvqphpsm'],
 ['ptpn', 'wqpe', 'vsuowfut', 'cxvxykkjk', 'xlxu', 'xpylmbi', 'xyvigkse'],
 ['begfz',
  'hyhonlr',
  'xpylmbi',
  'sbafp',
  'pyxjdwjjhjzwc',
  'xpylmbi',
  'lhlbk',
  'efxtky'],
 ['efxtky', 'jzinuxcc', 'vopoidjrt', 'wrkolipldqprfd'],
 ['tmxtudgn', 'jaocve', 'efxtky', 'lutghqtm', 'xpylmbi'],
 ['hqhxss', 'jzinuxcc', 'iqwiwmippjq

In [ ]:
from gensim.models import Word2Vec
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score
import numpy as np

model = Word2Vec(
    sentences=tokenized,
    vector_size=50,
    window=2,
    min_count=1,
    sg=0,
    workers=10,
    epochs=500
)

Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_fl

In [106]:
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC


def sentence_vectors(texts):
    means = []

    for sent in texts:
        embs = []

        for word in simple_preprocess(str(sent)):
            if word in model.wv:
                embs.append(model.wv[word])

        if embs:
            means.append(np.mean(embs, axis=0))
        else:
            means.append(np.zeros(model.vector_size))

    return np.array(means)


cl = make_pipeline(
    StandardScaler(),
    SVC(C=2.0, gamma="scale", class_weight="balanced", random_state=42),
)

X = sentence_vectors(train["text"])
y = train["label"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

cl.fit(X_train, y_train)
val_preds = cl.predict(X_test)
print("validation macro F1:", f1_score(y_test, val_preds, average="macro"))

# Train on all labeled rows before predicting the hidden/test rows.
cl.fit(X, y)
X_submit = sentence_vectors(test["text"])
preds = cl.predict(X_submit)

output = pd.DataFrame(columns=["subtaskID", "datapointID", "answer"])
output.loc[0] = [1, 0, len(words)]
output = pd.concat([output, pd.DataFrame({
    "subtaskID": 2,
    "datapointID": test["id"],
    "answer": preds
})], ignore_index=True)


In [107]:
output.to_csv("output.csv", index=False)

In [108]:
# window_size = 2
# pairs = []

# for sent in encoded:
#     for i, center in enumerate(sent):
#         start = max(0, i - window_size)
#         end = min(len(sent), i + window_size + 1)

#         for j in range(start, end):
#             if i != j:
#                 context = sent[j]
#                 pairs.append((center, context))

# pairs[:10]

In [109]:
# from torch.utils.data import DataLoader, Dataset
# import torch

# class Word2VecDataset(Dataset):
#     def __init__(self, pairs):
#         self.pairs = pairs

#     def __len__(self):
#         return len(self.pairs)
    
#     def __getitem__(self, idx):
#         center, context = self.pairs[idx]
#         return torch.tensor(center), torch.tensor(context)
    
# dataset = Word2VecDataset(pairs)
# loader = DataLoader(dataset, batch_size=16)

In [110]:
# from torch import nn

# class Word2VecModel(nn.Module): 
#     def __init__(self, vocab_size, embedding_dim):
#         super().__init__()

#         self.embedding = nn.Embedding(vocab_size, embedding_dim)
#         self.output = nn.Linear(embedding_dim, vocab_size)

#     def forward(self, center_word):
#         x = self.embedding(center_word)
#         return self.output(x)

In [111]:
# from torch import optim

# embedding_dim = 50
# epochs = 200

# model = Word2VecModel(len(vocab), embedding_dim)

# criterion = nn.CrossEntropyLoss()
# optimizer = optim.Adam(model.parameters(), lr=0.01)

# for epoch in range(epochs):
#     total_loss = 0

#     for center, context in loader:
#         logits = model(center)

#         loss = criterion(logits, context)
#         optimizer.zero_grad()
#         loss.backward()
#         optimizer.step()

#         total_loss += loss.item()

#     if epoch % 20 == 0:
#         print(f"Epoch {epoch}, loss: {total_loss:.4f}")